# AEM4L2 E04 - Tipos de errores ASR

**Objetivo:** identificar errores cotidianos de transcripción antes de medirlos con WER.

Este notebook es totalmente base: primero nombramos el tipo de error, después lo medimos.

**Python exercise relacionado:** `python_puro/AEM4_python_exercises/AEM4L2_audio_pipelines/e05_wer_error_critico.py`


## Los 4 resultados posibles al comparar palabras

| Reference | Hypothesis | Resultado |
|---|---|---|
| igual | igual | match |
| palabra A | palabra B | substitution |
| palabra | nada | deletion |
| nada | palabra extra | insertion |

Estos nombres son la base de WER.


In [ ]:
pairs = [
    ('pedido', 'pedido'),
    ('pedido', 'perdido'),
    ('urgente', None),
    (None, 'eh'),
]

for ref, hyp in pairs:
    if ref == hyp:
        label = 'match'
    elif ref is None:
        label = 'insertion'
    elif hyp is None:
        label = 'deletion'
    else:
        label = 'substitution'
    print(f'{str(ref):<10} | {str(hyp):<10} -> {label}')


## Substitution

Una palabra esperada fue reemplazada por otra.

Ejemplos cotidianos:

| Referencia | ASR | Impacto |
|---|---|---|
| pedido | perdido | cambia objeto del reclamo |
| cancelar | cambiar | cambia acción |
| ocho | dos | cambia dosis/frecuencia |
| vender | comprar | operación financiera contraria |


## Deletion

El ASR omite una palabra que sí estaba.

Ejemplos:

| Referencia | ASR | Qué se perdió |
|---|---|---|
| no autoricé el pago | autoricé el pago | negación crítica |
| entrega urgente hoy | entrega hoy | urgencia |
| cada ocho horas | cada horas | frecuencia |

La deletion puede ser más peligrosa cuando elimina negaciones o números.


## Insertion

El ASR agrega una palabra que no estaba.

Ejemplos:

| Referencia | ASR | Posible origen |
|---|---|---|
| quiero mi pedido | quiero eh mi pedido | muletilla/ruido |
| transferir mil pesos | transferir un mil pesos | normalización rara |
| cancelar pedido | cancelar el pedido urgente | palabra inferida |

A veces la insertion no cambia mucho. Otras veces introduce una acción o monto incorrecto.


In [ ]:
examples = [
    {'type': 'substitution', 'ref': 'cancelar', 'hyp': 'cambiar', 'risk': 'alto'},
    {'type': 'deletion', 'ref': 'no', 'hyp': '', 'risk': 'crítico'},
    {'type': 'insertion', 'ref': '', 'hyp': 'eh', 'risk': 'bajo'},
    {'type': 'substitution', 'ref': 'ocho', 'hyp': 'dos', 'risk': 'crítico'},
]

for e in examples:
    print(f"{e['type']:<13} ref={e['ref']!r:<10} hyp={e['hyp']!r:<10} riesgo={e['risk']}")


## Error de puntuación y segmentación

ASR no solo puede equivocarse en palabras. También puede cortar mal frases.

Ejemplo:

```text
Referencia: No, quiero cancelar.
ASR: No quiero cancelar.
```

La coma cambia mucho. En el segundo caso parece que la persona **no quiere cancelar**.

Esto muestra que a veces el problema no es una palabra sino la estructura de la frase.


## Error propagation

Un error ASR puede propagarse así:

| Etapa | Output |
|---|---|
| Audio real | "quiero cancelar mi pedido" |
| ASR | "quiero cambiar mi pedido" |
| LLM summary | "El cliente quiere modificar el pedido" |
| Backend | crea ticket `change_request` |
| Operación | agente intenta modificar, no cancelar |

Frase docente: **si la base está mal, lo demás puede verse prolijo pero estar equivocado.**


In [ ]:
def classify_intent(transcript: str) -> str:
    if 'cancelar' in transcript:
        return 'cancellation'
    if 'cambiar' in transcript:
        return 'change_request'
    return 'unknown'

for transcript in ['quiero cancelar mi pedido', 'quiero cambiar mi pedido']:
    print(transcript, '->', classify_intent(transcript))


## Pregunta de cierre

> ¿Qué error es peor: una insertion de `eh` o una deletion de `no`?

Respuesta esperada: depende del dominio, pero borrar una negación suele ser mucho más peligroso.
